# 05 — Random Forest (Baseline)
Best performing model — 72% test accuracy, Macro F1: 0.72.

Run `01_eda_preprocessing.ipynb` first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
np.random.seed(SEED)

In [ ]:
df = pd.read_csv('data/cleaned.csv')
feature_cols = [c for c in df.columns if c not in ['Customer_ID','Credit_Score_Enc']]
agg_df = df.groupby('Customer_ID')[feature_cols].mean()
target_df = df.groupby('Customer_ID')['Credit_Score_Enc'].agg(lambda x: x.mode()[0])
agg_df['target'] = target_df

X = StandardScaler().fit_transform(agg_df.drop(columns=['target']).values)
y = agg_df['target'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

In [ ]:
# Best config from presentation: 200 trees, max_depth=3, min_samples_leaf=2
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=3,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=SEED
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f'Random Forest Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred, target_names=['Poor','Standard','Good']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Poor','Standard','Good'], yticklabels=['Poor','Standard','Good'])
axes[0].set_title('Random Forest Confusion Matrix')

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=agg_df.drop(columns=['target']).columns)
feat_imp.nlargest(15).sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 Feature Importances')

plt.tight_layout()
plt.savefig('results/rf_results.png', dpi=150)
plt.show()